In [16]:

import sys
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split

from pathlib import Path

sys.path.append(str(Path().resolve().parent / 'src'))
from utils import backtest_dca_plus_trading_WITH_SL, backtest_dca_plus_trading_NO_SL, backtest_with_atr_SL,evaluate_all_strategies,backtest_dca_self_sufficient


In [2]:
with open('../models/best_models/xgboost_best_model.pkl', 'rb') as f: 
    ml_model = pickle.load(f)
with open('../models/models_num/scaler_num.pkl', 'rb') as f:
    scaler = pickle.load(f)

In [3]:
df=pd.read_csv('../data/VOO_ind_signal.csv')

In [5]:
features = [
    'rsi', 'macd_diff', 'bollinger_width', 'atr', 'adx', 
    'volume_ratio', 'price_vs_kijun', 'tenkan_vs_kijun'
]

In [6]:
X_final = df[features]
y_final = df['target']

In [7]:
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(X_final, y_final, test_size=0.2, shuffle=False)

In [ ]:
X_final_scaled = scaler.transform(X_final)
df['signal_ml'] = ml_model.predict(X_final_scaled)

In [9]:
df_backtest_test_only = df.loc[X_test_final.index]

In [24]:
#Parametros para los backtests 

estrategias_a_probar = [
    # Individuales
    'signal_ema_price',
    'signal_macd_buy',
    'signal_stochastic_buy',
    'signal_ichimoku_kijun_cross',
    'signal_bollinger_buy',
    
    # Combinadas
    'signal_macd_&_stochastic',
    'signal_ema_price_&_macd',
    'signal_ichimoku_kijun_cross_&_stochastic',
    'signal_ema_price_&_stochastic',
    
    #Machine Learning
    'signal_ml'    
]

params = {
    'monthly_invest': 400,
    'trade_amount': 150,
    'take_profit_pct': 0.125,
    'initial_trading_cash': 1000,
    'verbose': False 
}

params_atr = {
    'monthly_invest': 400,
    'trade_amount': 150,
    'atr_multiplier_tp': 10.0,   
    'atr_multiplier_sl': 3.0,  
    'initial_trading_cash': 1000,
    'verbose': False
}

In [25]:
print(" Iniciando Evaluación con Stops sin ATR")
df_resultados_finales = evaluate_all_strategies(df_backtest_test_only, estrategias_a_probar, backtest_dca_plus_trading_WITH_SL,params)

print("\n Tabla Comparativa de Estrategias")
display(df_resultados_finales)

 Iniciando Evaluación con Stops sin ATR
Evaluando: signal_ema_price...
Evaluando: signal_macd_buy...
Evaluando: signal_stochastic_buy...
Evaluando: signal_ichimoku_kijun_cross...
Evaluando: signal_bollinger_buy...
Evaluando: signal_macd_&_stochastic...
Evaluando: signal_ema_price_&_macd...
Evaluando: signal_ichimoku_kijun_cross_&_stochastic...
Evaluando: signal_ema_price_&_stochastic...
Evaluando: signal_ml...

 Tabla Comparativa de Estrategias


,Strategy,Final Portfolio Value,Total Contributions,Absolute Return,Percentage Return,Trading PnL,Trading Capital Used,Open Trades at End,Value of Open Trades,Sharpe Ratio,Total DCA Shares
0,signal_ml (With SL),18742.567574,15000,3742.567574,24.950450,306.796834,10200,7,1144.778426,2.723249,33.40543
1,signal_ichimoku_kijun_cross (With SL),18496.200563,15000,3496.200563,23.308004,137.255478,3750,2,317.952771,2.744159,33.40543
2,signal_macd_buy (With SL),18419.946897,15000,3419.946897,22.799646,78.954583,1050,0,0.000000,2.750340,33.40543
3,signal_ema_price (With SL),18381.251420,15000,3381.251420,22.541676,40.259106,750,0,0.000000,2.747249,33.40543
4,signal_stochastic_buy (With SL),18349.764490,15000,3349.764490,22.331763,8.772176,900,0,0.000000,2.744553,33.40543
5,signal_bollinger_buy (With SL),18340.992314,15000,3340.992314,22.273282,0.000000,0,0,0.000000,2.747167,33.40543
6,signal_macd_&_stochastic (With SL),18340.992314,15000,3340.992314,22.273282,0.000000,0,0,0.000000,2.747167,33.40543
7,signal_ema_price_&_macd (With SL),18340.992314,15000,3340.992314,22.273282,0.000000,0,0,0.000000,2.747167,33.40543
8,signal_ema_price_&_stochastic (With SL),18340.992314,15000,3340.992314,22.273282,0.000000,0,0,0.000000,2.747167,33.40543
9,signal_ichimoku_kijun_cross_&_stochastic (With...,18333.269157,15000,3333.269157,22.221794,-7.723156,150,0,0.000000,2.746190,33.40543


In [26]:
print(" Iniciando Evaluación con Stops con ATR")

df_results_atr = evaluate_all_strategies(df_backtest_test_only, estrategias_a_probar,  backtest_with_atr_SL, params_atr)
print("\n Tabla Comparativa con Stops con ATR ")
display(df_results_atr)

 Iniciando Evaluación con Stops con ATR
Evaluando: signal_ema_price...
Evaluando: signal_macd_buy...
Evaluando: signal_stochastic_buy...
Evaluando: signal_ichimoku_kijun_cross...
Evaluando: signal_bollinger_buy...
Evaluando: signal_macd_&_stochastic...
Evaluando: signal_ema_price_&_macd...
Evaluando: signal_ichimoku_kijun_cross_&_stochastic...
Evaluando: signal_ema_price_&_stochastic...
Evaluando: signal_ml...

 Tabla Comparativa con Stops con ATR 


,Strategy,Final Portfolio Value,Total Contributions,Absolute Return,Percentage Return,Trading PnL,Trading Capital Used,Open Trades at End,Value of Open Trades,Sharpe Ratio,Total DCA Shares
0,signal_ml (ATR Stops),18753.772222,15000,3753.772222,25.025148,296.992838,9900,8,1315.787071,2.716616,33.40543
1,signal_ichimoku_kijun_cross (ATR Stops),18508.060414,15000,3508.060414,23.387069,142.620788,3600,3,474.447313,2.743585,33.40543
2,signal_ema_price (ATR Stops),18414.561538,15000,3414.561538,22.763744,73.569224,750,0,0.000000,2.748392,33.40543
3,signal_macd_buy (ATR Stops),18411.478250,15000,3411.478250,22.743188,70.485936,1050,0,0.000000,2.748664,33.40543
4,signal_stochastic_buy (ATR Stops),18352.167436,15000,3352.167436,22.347783,11.175122,900,0,0.000000,2.745358,33.40543
5,signal_bollinger_buy (ATR Stops),18340.992314,15000,3340.992314,22.273282,0.000000,0,0,0.000000,2.747167,33.40543
6,signal_macd_&_stochastic (ATR Stops),18340.992314,15000,3340.992314,22.273282,0.000000,0,0,0.000000,2.747167,33.40543
7,signal_ema_price_&_macd (ATR Stops),18340.992314,15000,3340.992314,22.273282,0.000000,0,0,0.000000,2.747167,33.40543
8,signal_ema_price_&_stochastic (ATR Stops),18340.992314,15000,3340.992314,22.273282,0.000000,0,0,0.000000,2.747167,33.40543
9,signal_ichimoku_kijun_cross_&_stochastic (ATR ...,18336.152727,15000,3336.152727,22.241018,-4.839587,150,0,0.000000,2.746655,33.40543


In [27]:
print(" Iniciando Evaluación  sin Stops ni ATR")
df_resultados_finales = evaluate_all_strategies(df_backtest_test_only, estrategias_a_probar, backtest_dca_plus_trading_NO_SL,params)

print("\n Tabla Comparativa de Estrategias sin Stop ")
display(df_resultados_finales)

 Iniciando Evaluación  sin Stops ni ATR
Evaluando: signal_ema_price...
Evaluando: signal_macd_buy...
Evaluando: signal_stochastic_buy...
Evaluando: signal_ichimoku_kijun_cross...
Evaluando: signal_bollinger_buy...
Evaluando: signal_macd_&_stochastic...
Evaluando: signal_ema_price_&_macd...
Evaluando: signal_ichimoku_kijun_cross_&_stochastic...
Evaluando: signal_ema_price_&_stochastic...
Evaluando: signal_ml...

 Tabla Comparativa de Estrategias sin Stop 


,Strategy,Final Portfolio Value,Total Contributions,Absolute Return,Percentage Return,Trading PnL,Trading Capital Used,Open Trades at End,Value of Open Trades,Sharpe Ratio,Total DCA Shares
0,signal_ml (No SL),18800.869337,15000,3800.869337,25.339129,499.149564,5250,9,1310.727459,2.699639,33.40543
1,signal_ichimoku_kijun_cross (No SL),18657.308343,15000,3657.308343,24.382056,308.654269,3600,8,1207.661760,2.739827,33.40543
2,signal_stochastic_buy (No SL),18457.587354,15000,3457.587354,23.050582,116.595041,900,0,0.000000,2.750307,33.40543
3,signal_macd_buy (No SL),18449.292614,15000,3449.292614,22.995284,115.656091,1050,1,142.644209,2.750867,33.40543
4,signal_ema_price (No SL),18415.288269,15000,3415.288269,22.768588,76.243907,750,1,148.052048,2.748560,33.40543
5,signal_ichimoku_kijun_cross_&_stochastic (No SL),18361.351941,15000,3361.351941,22.409013,20.359627,150,0,0.000000,2.747692,33.40543
6,signal_macd_&_stochastic (No SL),18340.992314,15000,3340.992314,22.273282,0.000000,0,0,0.000000,2.747167,33.40543
7,signal_bollinger_buy (No SL),18340.992314,15000,3340.992314,22.273282,0.000000,0,0,0.000000,2.747167,33.40543
8,signal_ema_price_&_macd (No SL),18340.992314,15000,3340.992314,22.273282,0.000000,0,0,0.000000,2.747167,33.40543
9,signal_ema_price_&_stochastic (No SL),18340.992314,15000,3340.992314,22.273282,0.000000,0,0,0.000000,2.747167,33.40543


In [28]:
# Deshabilitar gráficos
params['plot'] = False
params['initial_trading_cash']=5000

df_resultados_finales = evaluate_all_strategies(df, estrategias_a_probar, backtest_dca_self_sufficient, params)

print("\n Tabla Comparativa de Estrategias sin Stop ")
display(df_resultados_finales)

Evaluando: signal_ema_price...
Evaluando: signal_macd_buy...
Evaluando: signal_stochastic_buy...
Evaluando: signal_ichimoku_kijun_cross...
Evaluando: signal_bollinger_buy...
Evaluando: signal_macd_&_stochastic...
Evaluando: signal_ema_price_&_macd...
Evaluando: signal_ichimoku_kijun_cross_&_stochastic...
Evaluando: signal_ema_price_&_stochastic...
Evaluando: signal_ml...

 Tabla Comparativa de Estrategias sin Stop 


,Strategy,Final Portfolio Value,Total Contributions,Absolute Return,Percentage Return,Trading PnL,Trading Capital Used,Open Trades at End,Value of Open Trades,Sharpe Ratio,DCA Total Shares,DCA Shares Value
0,signal_ml (Self-Sufficient),30457.313218,10600,19857.313218,187.333144,710.023957,6450,0,0,0.745774,58.460566,30347.289261
1,signal_macd_buy (Self-Sufficient),27381.011463,9800,17581.011463,179.398076,130.496580,2550,0,0,0.711492,52.109709,27050.514884
2,signal_stochastic_buy (Self-Sufficient),27312.522309,9800,17512.522309,178.699207,62.007425,2400,0,0,0.709417,52.109709,27050.514884
3,signal_ema_price (Self-Sufficient),27286.531458,9800,17486.531458,178.433994,36.016574,2700,0,0,0.707686,52.109709,27050.514884
4,signal_ema_price_&_macd (Self-Sufficient),27269.611460,9800,17469.611460,178.261341,19.096576,150,0,0,0.709591,52.109709,27050.514884
5,signal_ichimoku_kijun_cross_&_stochastic (Self...,27256.397275,9800,17456.397275,178.126503,5.882391,1350,0,0,0.708352,52.109709,27050.514884
6,signal_bollinger_buy (Self-Sufficient),27250.514884,9800,17450.514884,178.066478,0.000000,0,0,0,0.708693,52.109709,27050.514884
7,signal_macd_&_stochastic (Self-Sufficient),27250.514884,9800,17450.514884,178.066478,0.000000,0,0,0,0.708693,52.109709,27050.514884
8,signal_ema_price_&_stochastic (Self-Sufficient),27250.514884,9800,17450.514884,178.066478,0.000000,0,0,0,0.708693,52.109709,27050.514884
9,signal_ichimoku_kijun_cross (Self-Sufficient),27765.365474,10200,17565.365474,172.209465,244.354544,7200,0,0,0.715181,53.401343,27721.010929
